In [2]:
pip install matplotlib pandas

  Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl.metadata (6.6 kB)
  Using cached pillow-12.2.0-cp312-cp312-macosx_11_0_arm64.whl.metadata (8.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.1/8.1 MB 17.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 17.2 MB/s  0:00:00 eta 0:00:01
Using cached contourpy-1.3.3-cp312-cp312-macosx_11_0_arm64.whl (273 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.9/2.9 MB 18.3 MB/s  0:00:00
Using cached numpy-2.4.4-cp312-cp312-macosx_14_0_arm64.whl (5.2 MB)
Using cached pillow-12.2.0-cp312-cp312-macosx_11_0_arm64.whl (4.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9/9 [matplotlib]9 [matplotlib]

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pi

In [ ]:
RUN_SETS = {
    "e50": {
        "train": [
            ("logs/epochs/e50/train_run01.txt", "Run 01"),
            ("logs/epochs/e50/train_run02.txt", "Run 02"),
            ("logs/epochs/e50/train_run03.txt", "Run 03"),
        ],
        "eval": [
            ("logs/epochs/e50/eval_run01.txt", "Run 01"),
            ("logs/epochs/e50/eval_run02.txt", "Run 02"),
            ("logs/epochs/e50/eval_run03.txt", "Run 03"),
        ],
    },
    "e75": {
        "train": [
            ("logs/epochs/e75/train_run01.txt", "Run 01"),
            ("logs/epochs/e75/train_run02.txt", "Run 02"),
            ("logs/epochs/e75/train_run03.txt", "Run 03"),
        ],
        "eval": [
            ("logs/epochs/e75/eval_run01.txt", "Run 01"),
            ("logs/epochs/e75/eval_run02.txt", "Run 02"),
            ("logs/epochs/e75/eval_run03.txt", "Run 03"),
        ],
    },
    "lr2e5": {
        "train": [
            ("logs/lr/lr2e-5/train_run01.txt", "Run 01"),
            ("logs/lr/lr2e-5/train_run02.txt", "Run 02"),
            ("logs/lr/lr2e-5/train_run03.txt", "Run 03"),
        ],
        "eval": [
            ("logs/lr/lr2e-5/eval_run01.txt", "Run 01"),
            ("logs/lr/lr2e-5/eval_run02.txt", "Run 02"),
            ("logs/lr/lr2e-5/eval_run03.txt", "Run 03"),
        ],
    },
    "lr5e6": {
        "train": [
            ("logs/lr/lr5e-6/train_run01.txt", "Run 01"),
            ("logs/lr/lr5e-6/train_run02.txt", "Run 02"),
            ("logs/lr/lr5e-6/train_run03.txt", "Run 03"),
        ],
        "eval": [
            ("logs/lr/lr5e-6/eval_run01.txt", "Run 01"),
            ("logs/lr/lr5e-6/eval_run02.txt", "Run 02"),
            ("logs/lr/lr5e-6/eval_run03.txt", "Run 03"),
        ],
    },
}

SELECTED_RUN_SETS = ["e50", "lr2e5", "lr5e6"]

train_colors = ["#c25326", "#294069", "#f1b756"]
train_offsets = [(8, 8), (8, -28), (-70, 8)]
eval_set_colors = {
    "e50": "#c25326",
    "lr2e5": "#294069",
    "lr5e6": "#2d7f5e",
    "e75": "#9b59b6",
}


In [ ]:
import re
import matplotlib.pyplot as plt
import pandas as pd

TRAIN_EPOCH_RE = re.compile(
    r"Epoch\s+(\d+)/(\d+)\s+\|\s+loss=([0-9.]+)(?:\s+\|\s+silhouette=([0-9.]+))?"
)
EVAL_SECTION_RE = re.compile(r"===\s+(FROZEN_BASELINE|FINE_TUNED)\s+===")
EVAL_RANK_RE = re.compile(r"Rank-\s*(\d+) accuracy:\s*([0-9.]+)%")
EVAL_SIL_RE = re.compile(r"Silhouette score:\s*([-0-9.]+)")
CMC_RANKS = [1, 5, 10]


def parse_train_log(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            match = TRAIN_EPOCH_RE.search(line)
            if match:
                rows.append({
                    "epoch": int(match.group(1)),
                    "loss": float(match.group(3)),
                    "silhouette": float(match.group(4)) if match.group(4) else None,
                })
    return pd.DataFrame(rows)



def plot_training_runs(runs, colors=None, offsets=None, figsize=(14, 5)):
    colors = colors or ["#c25326", "#294069", "#f1b756"]
    offsets = offsets or [(8, 8), (8, -28), (-70, 8)]

    fig, axes = plt.subplots(1, 2, figsize=figsize)
    summary_rows = []

    for i, (path, title) in enumerate(runs):
        df = parse_train_log(path)
        if df.empty:
            print(f"No epoch data found in: {path}")
            continue

        xy_offset = offsets[i % len(offsets)]
        color = colors[i % len(colors)]

        axes[0].plot(df["epoch"], df["loss"], marker="o", markersize=4, label=title, color=color)

        best_loss_idx = df["loss"].idxmin()
        best_loss_epoch = df.loc[best_loss_idx, "epoch"]
        best_loss_value = df.loc[best_loss_idx, "loss"]

        axes[0].scatter(best_loss_epoch, best_loss_value, s=100, marker="*", zorder=5, color=color)
        axes[0].annotate(
            f"{title}\nbest: ep {best_loss_epoch}\n{best_loss_value:.4f}",
            (best_loss_epoch, best_loss_value),
            textcoords="offset points",
            xytext=xy_offset,
            fontsize=7,
            bbox=dict(boxstyle="round,pad=0.2", fc="#FFF7FB", ec=color, alpha=0.9),
            arrowprops=dict(arrowstyle="->", lw=0.8, alpha=0.6),
        )

        sil_df = df.dropna(subset=["silhouette"])
        best_sil_epoch = None
        best_sil_value = None

        if not sil_df.empty:
            axes[1].plot(sil_df["epoch"], sil_df["silhouette"], marker="o", markersize=4, label=title, color=color)
            best_sil_idx = sil_df["silhouette"].idxmax()
            best_sil_epoch = sil_df.loc[best_sil_idx, "epoch"]
            best_sil_value = sil_df.loc[best_sil_idx, "silhouette"]
            axes[1].scatter(best_sil_epoch, best_sil_value, s=100, marker="*", zorder=5, color=color)
            axes[1].annotate(
                f"{title}\nbest: ep {best_sil_epoch}\n{best_sil_value:.4f}",
                (best_sil_epoch, best_sil_value),
                textcoords="offset points",
                xytext=xy_offset,
                fontsize=7,
                bbox=dict(boxstyle="round,pad=0.2", fc="#FFF7FB", ec=color, alpha=0.9),
                arrowprops=dict(arrowstyle="->", lw=0.8, alpha=0.6),
            )

        summary_rows.append({
            "Run": title,
            "Best Loss Epoch": best_loss_epoch,
            "Best Loss": best_loss_value,
            "Best Silhouette Epoch": best_sil_epoch,
            "Best Silhouette": best_sil_value,
        })

    axes[0].set_title("Loss", fontsize=11)
    axes[0].set_xlabel("Epoch", fontsize=9)
    axes[0].set_ylabel("Loss", fontsize=9)
    axes[0].tick_params(axis="both", labelsize=8)
    axes[0].grid(True, alpha=0.3)
    axes[0].legend(fontsize=8)

    axes[1].set_title("Silhouette Score", fontsize=11)
    axes[1].set_xlabel("Epoch", fontsize=9)
    axes[1].set_ylabel("Silhouette", fontsize=9)
    axes[1].tick_params(axis="both", labelsize=8)
    axes[1].grid(True, alpha=0.3)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()

    return pd.DataFrame(summary_rows)



def print_train_summary(summary_df):
    print("\nBest values by run:")
    for _, row in summary_df.iterrows():
        print(
            f"{row['Run']}: best loss = {row['Best Loss']:.4f} at epoch {int(row['Best Loss Epoch'])}; "
            + (
                f"best silhouette = {row['Best Silhouette']:.4f} at epoch {int(row['Best Silhouette Epoch'])}"
                if pd.notna(row["Best Silhouette"])
                else "best silhouette = N/A"
            )
        )

    print("\nSummary table:")
    print(summary_df.to_string(index=False))

    loss_mean = summary_df["Best Loss"].mean()
    loss_std = summary_df["Best Loss"].std()
    sil_mean = summary_df["Best Silhouette"].mean()
    sil_std = summary_df["Best Silhouette"].std()
    loss_epoch_mean = summary_df["Best Loss Epoch"].mean()
    loss_epoch_std = summary_df["Best Loss Epoch"].std()
    sil_epoch_mean = summary_df["Best Silhouette Epoch"].mean()
    sil_epoch_std = summary_df["Best Silhouette Epoch"].std()

    print("\nMean ± standard deviation across runs:")
    print(f"Best Loss: {loss_mean:.4f} ± {loss_std:.4f}")
    print(f"Best Silhouette: {sil_mean:.4f} ± {sil_std:.4f}")
    print(f"Best Loss Epoch: {loss_epoch_mean:.2f} ± {loss_epoch_std:.2f}")
    print(f"Best Silhouette Epoch: {sil_epoch_mean:.2f} ± {sil_epoch_std:.2f}")



def analyze_training_runs(runs, colors=None, offsets=None, figsize=(14, 5)):
    summary_df = plot_training_runs(runs, colors=colors, offsets=offsets, figsize=figsize)
    print_train_summary(summary_df)
    return summary_df



def analyze_selected_train_runs(selected_run_sets, colors=None, offsets=None, figsize=(14, 5)):
    results = {}
    for run_set_name in selected_run_sets:
        print(f"\n{'=' * 20} {run_set_name} | Train {'=' * 20}")
        results[run_set_name] = analyze_training_runs(
            RUN_SETS[run_set_name]["train"],
            colors=colors,
            offsets=offsets,
            figsize=figsize,
        )
    return results



def parse_eval_log(path):
    sections = {
        "Frozen": {"Rank-1": None, "Rank-5": None, "Rank-10": None, "Silhouette": None},
        "Fine-tuned": {"Rank-1": None, "Rank-5": None, "Rank-10": None, "Silhouette": None},
    }
    current = None

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            section_match = EVAL_SECTION_RE.search(line)
            if section_match:
                current = "Frozen" if section_match.group(1) == "FROZEN_BASELINE" else "Fine-tuned"
                continue

            if current is None:
                continue

            rank_match = EVAL_RANK_RE.search(line)
            if rank_match:
                rank = int(rank_match.group(1))
                if rank in CMC_RANKS:
                    sections[current][f"Rank-{rank}"] = float(rank_match.group(2))
                continue

            sil_match = EVAL_SIL_RE.search(line)
            if sil_match:
                sections[current]["Silhouette"] = float(sil_match.group(1))

    return sections



def build_eval_summary(eval_runs):
    rows = []
    for path, title in eval_runs:
        metrics = parse_eval_log(path)
        rows.append({
            "Run": title,
            "Frozen Rank-1": metrics["Frozen"]["Rank-1"],
            "Frozen Rank-5": metrics["Frozen"]["Rank-5"],
            "Frozen Rank-10": metrics["Frozen"]["Rank-10"],
            "Frozen Silhouette": metrics["Frozen"]["Silhouette"],
            "Fine-tuned Rank-1": metrics["Fine-tuned"]["Rank-1"],
            "Fine-tuned Rank-5": metrics["Fine-tuned"]["Rank-5"],
            "Fine-tuned Rank-10": metrics["Fine-tuned"]["Rank-10"],
            "Fine-tuned Silhouette": metrics["Fine-tuned"]["Silhouette"],
        })
    return pd.DataFrame(rows)



def build_eval_aggregate(eval_summary_df):
    rows = []
    for condition in ["Frozen", "Fine-tuned"]:
        row = {"Condition": condition}
        for rank in CMC_RANKS:
            series = eval_summary_df[f"{condition} Rank-{rank}"]
            row[f"Rank-{rank} Mean"] = series.mean()
            row[f"Rank-{rank} Std"] = series.std()
        row["Silhouette Mean"] = eval_summary_df[f"{condition} Silhouette"].mean()
        row["Silhouette Std"] = eval_summary_df[f"{condition} Silhouette"].std()
        rows.append(row)
    return pd.DataFrame(rows)



def build_eval_comparison(selected_run_sets):
    rows = []
    aggregate_by_set = {}

    for run_set_name in selected_run_sets:
        eval_summary_df = build_eval_summary(RUN_SETS[run_set_name]["eval"])
        eval_aggregate_df = build_eval_aggregate(eval_summary_df)
        aggregate_by_set[run_set_name] = eval_aggregate_df

        for _, row in eval_aggregate_df.iterrows():
            rows.append({
                "Run Set": run_set_name,
                "Condition": row["Condition"],
                "Rank-1 Mean": row["Rank-1 Mean"],
                "Rank-1 Std": row["Rank-1 Std"],
                "Rank-5 Mean": row["Rank-5 Mean"],
                "Rank-5 Std": row["Rank-5 Std"],
                "Rank-10 Mean": row["Rank-10 Mean"],
                "Rank-10 Std": row["Rank-10 Std"],
                "Silhouette Mean": row["Silhouette Mean"],
                "Silhouette Std": row["Silhouette Std"],
            })

    return pd.DataFrame(rows), aggregate_by_set



def plot_eval_comparison(eval_comparison_df, set_colors=None):
    set_colors = set_colors or {}
    plt.figure(figsize=(8, 5))

    for run_set_name in eval_comparison_df["Run Set"].unique():
        subset = eval_comparison_df[eval_comparison_df["Run Set"] == run_set_name]
        color = set_colors.get(run_set_name, None)

        for condition, linestyle in [("Frozen", "--"), ("Fine-tuned", "-")]:
            row = subset[subset["Condition"] == condition].iloc[0]
            means = [row[f"Rank-{rank} Mean"] for rank in CMC_RANKS]
            stds = [row[f"Rank-{rank} Std"] for rank in CMC_RANKS]
            plt.plot(
                CMC_RANKS,
                means,
                marker="o",
                linewidth=2,
                linestyle=linestyle,
                color=color,
                label=f"{run_set_name} | {condition}",
            )
            plt.fill_between(
                CMC_RANKS,
                [m - s for m, s in zip(means, stds)],
                [m + s for m, s in zip(means, stds)],
                color=color,
                alpha=0.10,
            )

    plt.xticks(CMC_RANKS)
    plt.ylim(0, 100)
    plt.xlabel("CMC Rank")
    plt.ylabel("Accuracy (%)")
    plt.title("Eval Comparison Across Run Sets")
    plt.grid(True, alpha=0.3)
    plt.legend(fontsize=8)
    plt.tight_layout()
    plt.show()



def print_eval_comparison(eval_comparison_df):
    print("\nEval comparison table (mean ± std across runs in each set):")
    print(eval_comparison_df.to_string(index=False))

    for run_set_name in eval_comparison_df["Run Set"].unique():
        print(f"\n{run_set_name}:")
        subset = eval_comparison_df[eval_comparison_df["Run Set"] == run_set_name]
        for _, row in subset.iterrows():
            print(
                f"{row['Condition']}: "
                f"Rank-1 = {row['Rank-1 Mean']:.2f} ± {row['Rank-1 Std']:.2f}, "
                f"Rank-5 = {row['Rank-5 Mean']:.2f} ± {row['Rank-5 Std']:.2f}, "
                f"Rank-10 = {row['Rank-10 Mean']:.2f} ± {row['Rank-10 Std']:.2f}, "
                f"Silhouette = {row['Silhouette Mean']:.4f} ± {row['Silhouette Std']:.4f}"
            )



def analyze_eval_run_sets(selected_run_sets, set_colors=None):
    eval_comparison_df, aggregate_by_set = build_eval_comparison(selected_run_sets)
    plot_eval_comparison(eval_comparison_df, set_colors=set_colors)
    print_eval_comparison(eval_comparison_df)
    return eval_comparison_df, aggregate_by_set



def main(selected_run_sets=None, train_figsize=(14, 5)):
    if selected_run_sets is None:
        selected_run_sets = SELECTED_RUN_SETS

    train_results = analyze_selected_train_runs(
        selected_run_sets,
        colors=train_colors,
        offsets=train_offsets,
        figsize=train_figsize,
    )

    print(f"\n{'=' * 20} Eval Comparison {'=' * 20}")
    eval_comparison_df, eval_aggregate_by_set = analyze_eval_run_sets(
        selected_run_sets,
        set_colors=eval_set_colors,
    )

    return {
        "train_results": train_results,
        "eval_comparison_df": eval_comparison_df,
        "eval_aggregate_by_set": eval_aggregate_by_set,
    }



In [ ]:
results = main(selected_run_sets=SELECTED_RUN_SETS)
results["eval_comparison_df"]
